<a href="https://colab.research.google.com/github/Bee-Ene/shri-qlearning-execute-or-clarify/blob/main/BaselineComp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Baseline Comparison Experiment

In [ ]:
'''
Compares three policies across easy, mid, and hard environments:
1. Q-Learning Agent: the trained policy
2. Random Policy:    selects EXECUTE or CLARIFY with equal probability
3. Rule-Based Policy:EXECUTE if COMPLETE+CLEAR, CLARIFY otherwise
'''

import numpy as np
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv
import os
from enum import Enum
from typing import Optional, Tuple
from dataclasses import dataclass
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#STATE SPACE (same as the one used)

class Completeness(Enum):
    COMPLETE   = 0
    PARTIAL    = 1
    INCOMPLETE = 2

class Clarity(Enum):
    CLEAR     = 0
    AMBIGUOUS = 1

class Feasibility(Enum):
    FEASIBLE  = 0
    UNCERTAIN = 1
    BLOCKED   = 2

UNAMBIGUOUS_INTENTS = {"STOP", "SIT", "STAND"}

PARSE_AMBIGUOUS_THRESHOLD = 0.60
PERC_UNCERTAIN_THRESHOLD  = 0.45
PERC_UNCERTAIN_NAV        = 0.50

def encode_state(c, cl, f): return c.value * 6 + cl.value * 3 + f.value
def decode_state(s):
    return Completeness(s//6), Clarity((s%6)//3), Feasibility(s%3)

NUM_STATES  = 18
NUM_ACTIONS = 2
ALL_STATES  = list(range(NUM_STATES))

def state_uncertainty_level(s):
    c, cl, f = decode_state(s)
    score = 0
    if c == Completeness.INCOMPLETE: score += 2
    elif c == Completeness.PARTIAL:  score += 1
    if cl == Clarity.AMBIGUOUS:      score += 1
    if f == Feasibility.BLOCKED:     score += 2
    elif f == Feasibility.UNCERTAIN: score += 1
    return score

def oracle_should_clarify(c, cl, f):
    return not (c == Completeness.COMPLETE and cl == Clarity.CLEAR and f == Feasibility.FEASIBLE)


# DOMAIN (same as before)

INTENTS = ["MOVE", "TURN", "NAVIGATE_TO", "STOP", "SIT", "STAND"]
REQUIRED_PARAMS = {
    "MOVE": ["direction"], "TURN": ["direction"],
    "NAVIGATE_TO": ["target"], "STOP": [], "SIT": [], "STAND": [],
}
DIRECTIONS = ["forward", "backward", "left", "right"]
TARGETS    = ["the chair", "the door", "the table", "the person", "the marker"]

@dataclass
class LocoCommand:
    intent: str; direction: Optional[str]; target: Optional[str]
    distance: Optional[float]; raw_text: str; parse_confidence: float
    present_params: list; missing_params: list

@dataclass
class PerceptionState:
    path_clear: bool; target_visible: bool; num_target_matches: int
    obstacle_distance: float; perception_confidence: float


# SIMULATION (same)

def simulate_command_parser(scenario, noise):
    intent    = scenario["intent"]
    required  = REQUIRED_PARAMS[intent]
    direction = scenario.get("direction")
    target    = scenario.get("target")
    present_params = []; missing_params = []
    if "direction" in required:
        if direction and random.random() < noise: direction = None
        (present_params if direction is not None else missing_params).append("direction")
    if "target" in required:
        if target and random.random() < noise: target = None
        (present_params if target is not None else missing_params).append("target")
    if intent in UNAMBIGUOUS_INTENTS:
        conf = float(np.clip(0.92 + random.gauss(0, 0.02), 0.85, 0.99))
    else:
        base = 0.90 - (0.28 * noise) - (0.10 * len(missing_params))
        conf = float(np.clip(base + random.gauss(0, 0.04), 0.05, 0.99))
    parts = [intent.lower().replace("_", " ")]
    if direction: parts.append(direction)
    if target:    parts.append(f"to {target}")
    return LocoCommand(intent=intent, direction=direction, target=target,
                       distance=None, raw_text=" ".join(parts),
                       parse_confidence=conf, present_params=present_params,
                       missing_params=missing_params)

def simulate_perception(scenario, noise):
    num_obs = scenario.get("num_obstacles", 0)
    num_tgt = scenario.get("num_targets", 1)
    if random.random() < noise * 0.3: num_obs = max(0, num_obs + random.choice([-1,1]))
    if random.random() < noise * 0.35: num_tgt = max(0, num_tgt + random.choice([-1,1]))
    path_clear = num_obs == 0
    obs_dist   = float(np.clip(random.gauss(0.8, 0.2), 0.3, 1.5)) if not path_clear else float('inf')
    if path_clear and num_tgt == 1: base = 0.88
    elif not path_clear or num_tgt == 0: base = 0.28
    else: base = 0.52
    perc_conf = float(np.clip(base + random.gauss(0, 0.06), 0.05, 0.98))
    return PerceptionState(path_clear=path_clear, target_visible=num_tgt>0,
                           num_target_matches=num_tgt, obstacle_distance=obs_dist,
                           perception_confidence=perc_conf)

def build_state(cmd, perc):
    required = REQUIRED_PARAMS[cmd.intent]
    if not required: comp = Completeness.COMPLETE
    else:
        n_pres = len(cmd.present_params); n_req = len(required)
        if n_pres == n_req:   comp = Completeness.COMPLETE
        elif n_pres == 0:     comp = Completeness.INCOMPLETE
        else:                 comp = Completeness.PARTIAL
    if cmd.intent in UNAMBIGUOUS_INTENTS: clar = Clarity.CLEAR
    else: clar = Clarity.AMBIGUOUS if cmd.parse_confidence < PARSE_AMBIGUOUS_THRESHOLD else Clarity.CLEAR
    needs_target = cmd.intent == "NAVIGATE_TO"
    if not perc.path_clear: feas = Feasibility.BLOCKED
    elif needs_target:
        if not perc.target_visible: feas = Feasibility.BLOCKED
        elif perc.num_target_matches > 1: feas = Feasibility.UNCERTAIN
        elif perc.perception_confidence < PERC_UNCERTAIN_NAV: feas = Feasibility.UNCERTAIN
        else: feas = Feasibility.FEASIBLE
    else: feas = Feasibility.UNCERTAIN if perc.perception_confidence < PERC_UNCERTAIN_THRESHOLD else Feasibility.FEASIBLE
    return encode_state(comp, clar, feas)

def generate_scenario(difficulty):
    cfg = {
        "easy": {"drop_prob": 0.10, "obs_pool": [0,0,0,1], "tgt_pool": [1,1,1,2]},
        "mid":  {"drop_prob": 0.25, "obs_pool": [0,0,1,1,2], "tgt_pool": [0,1,1,2]},
        "hard": {"drop_prob": 0.42, "obs_pool": [0,1,1,2,2], "tgt_pool": [0,0,1,2,3]},
    }[difficulty]
    intent    = random.choice(INTENTS)
    needed    = REQUIRED_PARAMS[intent]
    direction = random.choice(DIRECTIONS) if "direction" in needed and random.random() > cfg["drop_prob"] else None
    target    = random.choice(TARGETS)    if "target"    in needed and random.random() > cfg["drop_prob"] else None
    return {
        "intent": intent, "direction": direction, "target": target, "distance": None,
        "num_obstacles": random.choice(cfg["obs_pool"]),
        "num_targets":   random.choice(cfg["tgt_pool"]) if "target" in needed else 0,
    }


# EXECUTION SUCCESS AND REWARD (same)


COMPLETENESS_FACTOR = {Completeness.COMPLETE: 1.00, Completeness.PARTIAL: 0.72, Completeness.INCOMPLETE: 0.38}
CLARITY_FACTOR      = {Clarity.CLEAR: 1.00, Clarity.AMBIGUOUS: 0.60}
FEASIBILITY_FACTOR  = {Feasibility.FEASIBLE: 1.00, Feasibility.UNCERTAIN: 0.75, Feasibility.BLOCKED: 0.15}
BASE_SUCCESS        = 0.95

def exec_success_prob(state):
    c, cl, f = decode_state(state)
    return float(np.clip(BASE_SUCCESS * COMPLETENESS_FACTOR[c] * CLARITY_FACTOR[cl] * FEASIBILITY_FACTOR[f], 0.02, 0.99))

REWARD_EXEC_SUCCESS      = +10.0
REWARD_EXEC_FAIL         = -10.0
REWARD_CLAR_PRODUCTIVE   =  +3.0
REWARD_CLAR_UNPRODUCTIVE =  -2.0
REWARD_STEP_COST         =  -1.0
REWARD_EXCESSIVE_CLAR    =  -5.0


# THREE POLICY IMPLEMENTATIONS

class PolicyType:
    Q_LEARNING = "Q-Learning"
    RANDOM     = "Random"
    RULE_BASED = "Rule-Based"

def select_action_qlearning(Q, state):
    #Greedy Q-table lookup.
    return int(np.argmax(Q[state]))

def select_action_random(state):
    #Equal probability random selection (baseline lower bound).
    return random.randint(0, 1)

def select_action_rulebased(state):
    '''
    Deterministic rule: EXECUTE only when COMPLETE+CLEAR (feasibility ignored).
    For a non-linear system, it uses this method.
    CLARIFY whenever completeness or clarity is not perfect.
    '''
    c, cl, f = decode_state(state)
    if c == Completeness.COMPLETE and cl == Clarity.CLEAR:
        return 0  # EXECUTE
    return 1  # CLARIFY


# SINGLE EPISODE RUNNER (policy-agnostic)

def run_episode(policy_type, Q, difficulty, exec_scale, clarify_success, noise):
    '''
    Runs one episode under the given policy.
    Q is only used when policy_type == PolicyType.Q_LEARNING.
    All other logic (rewards, state transitions), identical across policies.
    This ensures a fair comparison. Only the action selection differs.
    '''
    MAX_CLAR = 3
    scenario = generate_scenario(difficulty)
    total_reward = 0.0
    n_clarify    = 0
    n_productive = 0
    exec_success = False
    did_execute  = False

    for step in range(MAX_CLAR + 1):
        cmd  = simulate_command_parser(scenario, noise)
        perc = simulate_perception(scenario, noise)
        state = build_state(cmd, perc)

        # Select action according to policy
        if policy_type == PolicyType.Q_LEARNING:
            action = select_action_qlearning(Q, state)
        elif policy_type == PolicyType.RANDOM:
            action = select_action_random(state)
        else:
            action = select_action_rulebased(state)

        if action == 0:  # EXECUTE
            prob    = min(exec_success_prob(state) * exec_scale, 0.99)
            success = random.random() < prob
            reward  = REWARD_EXEC_SUCCESS if success else REWARD_EXEC_FAIL
            total_reward += reward
            exec_success  = success
            did_execute   = True
            break
        else:  # CLARIFY
            if step == MAX_CLAR:
                total_reward += REWARD_EXCESSIVE_CLAR
                break
            n_clarify += 1
            resolution = random.random() < clarify_success
            if resolution:
                scenario["direction"]     = scenario.get("direction") or random.choice(DIRECTIONS)
                scenario["target"]        = scenario.get("target")    or random.choice(TARGETS)
                scenario["num_obstacles"] = 0
                scenario["num_targets"]   = 1
            next_noise = noise * 0.55 if resolution else noise
            cmd_next   = simulate_command_parser(scenario, next_noise)
            perc_next  = simulate_perception(scenario, next_noise)
            next_s     = build_state(cmd_next, perc_next)
            if state_uncertainty_level(next_s) < state_uncertainty_level(state):
                n_productive += 1
                reward = REWARD_STEP_COST + REWARD_CLAR_PRODUCTIVE
            else:
                reward = REWARD_STEP_COST + REWARD_CLAR_UNPRODUCTIVE
            total_reward += reward

    clar_precision = (n_productive / n_clarify) if n_clarify > 0 else None
    return {
        "total_reward":   total_reward,
        "exec_success":   exec_success,
        "did_execute":    did_execute,
        "n_clarify":      n_clarify,
        "n_productive":   n_productive,
        "clar_precision": clar_precision,
    }


# EVALUATION RUNNER


def evaluate_policy(policy_type, Q, difficulty, exec_scale, clarify_success,
                    noise, n_episodes=1000):
    exec_ok = exec_fail = clar_tot = clar_prod = excess = 0
    rewards = []
    for _ in range(n_episodes):
        result = run_episode(policy_type, Q, difficulty, exec_scale, clarify_success, noise)
        if result["did_execute"]:
            if result["exec_success"]: exec_ok   += 1
            else:                      exec_fail += 1
        else:
            excess += 1
        clar_tot  += result["n_clarify"]
        clar_prod += result["n_productive"]
        rewards.append(result["total_reward"])
    te = exec_ok + exec_fail + 1e-9
    tc = clar_tot
    return {
        "task_success_rate":            round(exec_ok  / te, 4),
        "failure_rate":                 round(exec_fail / te, 4),
        "clarification_rate":           round(clar_tot / n_episodes, 3),
        "clarification_precision":      round(clar_prod / tc, 4) if tc > 0 else None,
        "excessive_clarification_rate": round(excess / n_episodes, 4),
        "mean_episode_reward":          round(float(np.mean(rewards)), 3),
    }

def run_comparison(policy_type, Q, difficulty, exec_scale, clarify_success,
                   noise, n_runs=10, n_episodes=1000):
    all_metrics = []
    for seed_i in range(n_runs):
        random.seed(SEED + seed_i)
        np.random.seed(SEED + seed_i)
        metrics = evaluate_policy(policy_type, Q, difficulty, exec_scale,
                                  clarify_success, noise, n_episodes)
        all_metrics.append(metrics)
    random.seed(SEED); np.random.seed(SEED)

    numeric_keys = [k for k in all_metrics[0] if isinstance(all_metrics[0][k], float)]
    avg = {k: round(float(np.mean([m[k] for m in all_metrics if m[k] is not None])), 4)
           for k in numeric_keys}
    std = {k: round(float(np.std( [m[k] for m in all_metrics if m[k] is not None])), 4)
           for k in numeric_keys}
    return {"avg": avg, "std": std}


# VISUALISATIONS


def plot_comparison_bars(results_by_env, results_path):
    '''
    Grouped bar chart: for each environment, show all three policies
    side by side on key metrics.
    '''
    envs       = ["easy", "mid", "hard"]
    policies   = [PolicyType.Q_LEARNING, PolicyType.RANDOM, PolicyType.RULE_BASED]
    colors     = {"Q-Learning": "#2563eb", "Random": "#dc2626", "Rule-Based": "#16a34a"}
    metrics    = ["task_success_rate", "clarification_precision",
                  "excessive_clarification_rate", "mean_episode_reward"]
    m_labels   = ["Task Success Rate", "Clarification Precision",
                  "Excessive Clarif. Rate", "Mean Episode Reward"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    axes = axes.flatten()

    for mi, (metric, label) in enumerate(zip(metrics, m_labels)):
        ax   = axes[mi]
        x    = np.arange(len(envs))
        w    = 0.25
        for pi, policy in enumerate(policies):
            vals = [results_by_env[env][policy]["avg"].get(metric, 0) for env in envs]
            errs = [results_by_env[env][policy]["std"].get(metric, 0) for env in envs]
            ax.bar(x + pi*w, vals, w, label=policy, color=colors[policy],
                   alpha=0.85, capsize=4, ecolor="#374151",
                   yerr=errs if metric != "mean_episode_reward" else None)
        ax.set_xticks(x + w)
        ax.set_xticklabels(["Easy", "Mid", "Hard"], fontsize=11)
        ax.set_title(label, fontsize=12, fontweight="bold")
        ax.set_ylabel("Score", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.2)

    plt.suptitle("Policy Comparison: Q-Learning vs Random vs Rule-Based\n"
                 "(Mean ± Std, 10 seeds, 1000 eval episodes per seed)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(results_path, "baseline_comparison_bars.png"),
                dpi=200, bbox_inches="tight")
    plt.close()
    print("  Saved: baseline_comparison_bars.png")


def plot_comparison_radar(results_by_env, results_path):
    '''
    Radar chart showing overall policy profile across all environments combined.
    Each axis is a metric.
    '''
    from matplotlib.patches import FancyArrowPatch

    policies = [PolicyType.Q_LEARNING, PolicyType.RANDOM, PolicyType.RULE_BASED]
    colors   = {"Q-Learning": "#2563eb", "Random": "#dc2626", "Rule-Based": "#16a34a"}
    metrics  = ["task_success_rate", "clarification_precision",
                "mean_episode_reward_norm", "low_excess_clarif"]
    m_labels = ["Task\nSuccess Rate", "Clarification\nPrecision",
                "Episode\nReward\n(normalised)", "Low Excess\nClarif."]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ei, env in enumerate(["easy", "mid", "hard"]):
        ax = axes[ei]
        ax.set_aspect("equal")

        # Compute normalised values for each policy
        for policy in policies:
            avg = results_by_env[env][policy]["avg"]
            # Normalise reward to [0,1] range (min=-5, max=+12 approx)
            reward_norm = (avg.get("mean_episode_reward", 0) + 5) / 17
            reward_norm = np.clip(reward_norm, 0, 1)
            # Invert excessive clarification (lower is better -> higher score)
            low_excess  = 1.0 - avg.get("excessive_clarification_rate", 0)

            vals = [
                avg.get("task_success_rate", 0),
                avg.get("clarification_precision", 0),
                reward_norm,
                low_excess,
            ]

            N    = len(vals)
            angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
            vals_plot   = vals + [vals[0]]
            angles_plot = angles + [angles[0]]

            ax.plot(angles_plot, vals_plot, color=colors[policy],
                    linewidth=2.2, label=policy, alpha=0.85)
            ax.fill(angles_plot, vals_plot, color=colors[policy], alpha=0.08)

        ax.set_xticks(angles)
        ax.set_xticklabels(m_labels, size=8)
        ax.set_ylim(0, 1.1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["0.25","0.5","0.75","1.0"], size=7)
        ax.set_title(env.capitalize() + " Environment",
                     fontsize=11, fontweight="bold", pad=15)
        if ei == 1:
            ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.15), fontsize=9)

    plt.suptitle("Policy Profile Radar: Q-Learning vs Baselines",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(results_path, "baseline_comparison_radar.png"),
                dpi=200, bbox_inches="tight")
    plt.close()
    print("  Saved: baseline_comparison_radar.png")


def plot_policy_decision_table(results_path):
    '''
    Visual table showing what each policy decides for all 18 states.
    Reveals qualitative differences between policies.
    '''
    q_path = os.path.join(results_path, "mid_q_table.npy")
    if not os.path.exists(q_path):
        print("  Skipping policy decision table (mid_q_table.npy not found)")
        return
    Q = np.load(q_path)

    decisions = np.zeros((18, 3))  # cols: Q-Learning, Random, Rule-Based
    for s in range(18):
        decisions[s, 0] = int(np.argmax(Q[s]))
        decisions[s, 1] = 0.5  # random always 50/50
        decisions[s, 2] = select_action_rulebased(s)

    state_labels = []
    for s in range(18):
        c, cl, f = decode_state(s)
        state_labels.append(f"[{s}] {c.name[:4]}\n{cl.name[:4]}\n{f.name[:4]}")

    fig, ax = plt.subplots(figsize=(10, 14))
    cmap = plt.cm.RdBu_r
    im   = ax.imshow(decisions, cmap=cmap, vmin=0, vmax=1, aspect="auto")

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Q-Learning\n(trained)", "Random\n(50/50)", "Rule-Based\n(Complete+Clear→Exec)"],
                       fontsize=10, fontweight="bold")
    ax.set_yticks(range(18))
    ax.set_yticklabels(state_labels, fontsize=8)

    for s in range(18):
        for pi in range(3):
            val = decisions[s, pi]
            if pi == 1:
                txt = "50/50"
                color = "black"
            else:
                txt   = "EXEC" if val == 0 else "CLAR"
                color = "#f9fafb" if val > 0.5 else "#111827"
            ax.text(pi, s, txt, ha="center", va="center",
                    fontsize=8, fontweight="bold", color=color)

    plt.colorbar(im, ax=ax, label="Action (0=EXECUTE, 1=CLARIFY)", shrink=0.6)
    ax.set_title("Decision Comparison Across All 18 States\n"
                 "Q-Learning (mid Q-table) vs Rule-Based vs Random",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(results_path, "baseline_decision_table.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: baseline_decision_table.png")



# MAIN


if __name__ == "__main__":

    BASE_PATH   = "/PATH"

    # Find the most recent run folder that contains q-tables
    run_folders = sorted([
        os.path.join(BASE_PATH, d)
        for d in os.listdir(BASE_PATH)
        if d.startswith("run_") and os.path.isdir(os.path.join(BASE_PATH, d))
    ])
    if not run_folders:
        raise FileNotFoundError(
            f"No run_ folders found in {BASE_PATH}. "
            "Run shri_qlearning_FINAL.py first.")

    QTABLE_PATH   = run_folders[-1]   # most recent run
    RESULTS_PATH  = os.path.join(BASE_PATH,
                                 f"baseline_comparison_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")
    os.makedirs(RESULTS_PATH, exist_ok=True)

    # Copy q-tables from most recent run to results path
    import shutil
    for env in ["easy", "mid", "hard"]:
        src = os.path.join(QTABLE_PATH, f"{env}_q_table.npy")
        dst = os.path.join(RESULTS_PATH, f"{env}_q_table.npy")
        if os.path.exists(src):
            shutil.copy(src, dst)

    print("=" * 65)
    print("  SHRI: Baseline Comparison Experiment")
    print("  Comparing: Q-Learning vs Random vs Rule-Based")
    print(f"  Q-tables from: {QTABLE_PATH}")
    print(f"  Results → {RESULTS_PATH}")
    print("=" * 65)

    # Environment configurations (same as before)
    environments = {
        "easy": {"exec_scale": 1.10, "clarify_success": 0.90, "noise": 0.08},
        "mid":  {"exec_scale": 1.00, "clarify_success": 0.75, "noise": 0.15},
        "hard": {"exec_scale": 0.85, "clarify_success": 0.60, "noise": 0.25},
    }

    policies      = [PolicyType.Q_LEARNING, PolicyType.RANDOM, PolicyType.RULE_BASED]
    all_results   = {env: {} for env in environments}
    summary_rows  = []

    for env_name, cfg in environments.items():
        print(f"\n{'─'*65}")
        print(f"  Environment: {env_name.upper()}")
        print(f"{'─'*65}")

        # Load the Q-table trained for this environment
        q_path = os.path.join(RESULTS_PATH, f"{env_name}_q_table.npy")
        if not os.path.exists(q_path):
            print(f"  WARNING: {env_name}_q_table.npy not found. "
                  f"Using zero Q-table for Q-Learning in this environment.")
            Q = np.zeros((NUM_STATES, NUM_ACTIONS))
        else:
            Q = np.load(q_path)
            print(f"  Q-table loaded: shape {Q.shape}")

        for policy in policies:
            print(f"\n  Policy: {policy}")
            result = run_comparison(
                policy_type     = policy,
                Q               = Q,
                difficulty      = env_name,
                exec_scale      = cfg["exec_scale"],
                clarify_success = cfg["clarify_success"],
                noise           = cfg["noise"],
                n_runs          = 10,
                n_episodes      = 1000,
            )
            all_results[env_name][policy] = result
            print(f"    task_success_rate:            {result['avg'].get('task_success_rate',0):.4f} "
                  f"± {result['std'].get('task_success_rate',0):.4f}")
            print(f"    clarification_precision:      {result['avg'].get('clarification_precision',0):.4f} "
                  f"± {result['std'].get('clarification_precision',0):.4f}")
            print(f"    excessive_clarification_rate: {result['avg'].get('excessive_clarification_rate',0):.4f} "
                  f"± {result['std'].get('excessive_clarification_rate',0):.4f}")
            print(f"    mean_episode_reward:          {result['avg'].get('mean_episode_reward',0):.3f} "
                  f"± {result['std'].get('mean_episode_reward',0):.3f}")

            summary_rows.append({
                "environment": env_name,
                "policy": policy,
                **{f"avg_{k}": v for k, v in result["avg"].items()},
                **{f"std_{k}": v for k, v in result["std"].items()},
            })

    # Summary table
    print(f"\n{'='*65}")
    print("  SUMMARY: Q-Learning vs Baselines")
    print(f"{'='*65}")
    print(f"  {'Env':<6} {'Policy':<12} {'Success':>9} {'Clarif Prec':>12} {'Excess Clar':>12} {'Reward':>9}")
    print(f"  {'─'*64}")
    for env_name in ["easy", "mid", "hard"]:
        for policy in policies:
            r = all_results[env_name][policy]["avg"]
            print(f"  {env_name:<6} {policy:<12} "
                  f"{r.get('task_success_rate',0):>9.4f} "
                  f"{r.get('clarification_precision',0):>12.4f} "
                  f"{r.get('excessive_clarification_rate',0):>12.4f} "
                  f"{r.get('mean_episode_reward',0):>9.3f}")

    # Save results
    print(f"\n  Generating plots...")
    plot_comparison_bars(all_results, RESULTS_PATH)
    plot_comparison_radar(all_results, RESULTS_PATH)
    plot_policy_decision_table(RESULTS_PATH)

    # Save CSV
    if summary_rows:
        csv_path = os.path.join(RESULTS_PATH, "baseline_comparison_results.csv")
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
            writer.writeheader()
            writer.writerows(summary_rows)
        print(f"  Results CSV saved: baseline_comparison_results.csv")

    print(f"\n{'='*65}")
    print("  Baseline comparison complete.")
    print(f"  Results saved to: {RESULTS_PATH}")
    print(f"{'='*65}")